# 📈 Banco de Dados de Séries Temporais (Time-Series)
> **Missão:** Demonstrar a altíssima performance de consultar dados fatiados no tempo (Janela Temporal) versus o erro fatal de buscar dados métricos pelo valor global.
> **Arquivo:** Simularemos 1 milhão de logs de sensores de um servidor salvos em `telemetria_servidor.csv`.

Sistemas de telemetria e IoT geram montanhas de dados por segundo. Em um banco Time-Series, o disco é organizado cronologicamente. Filtrar o que aconteceu "na última hora" é quase instantâneo. Mas procurar "todas as vezes em que a CPU bateu 99%" na história inteira derruba o servidor.

In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

file_path = 'telemetria_servidor.csv'

if not os.path.exists(file_path):
    print("⏳ Gerando 'telemetria_servidor.csv' no disco (isso pode levar uns 10 a 15 segundos)...")
    
    # Gerando 1 milhão de segundos (aprox. 11.5 dias de logs ininterruptos)
    datas = pd.date_range(start='2026-01-01 00:00:00', periods=1000000, freq='s') 
    
    # Gerando métricas sintéticas
    np.random.seed(42)
    cpu = np.random.normal(loc=45.0, scale=15.0, size=1000000).clip(0, 100) # CPU entre 0 e 100%
    temp = np.random.normal(loc=60.0, scale=10.0, size=1000000) # Temperatura
    
    df_ts = pd.DataFrame({'timestamp': datas, 'cpu_usage': cpu, 'temperatura': temp})
    
    # Salvando no disco
    df_ts.to_csv(file_path, index=False)
        
    print(f"✅ Arquivo salvo! Foram gerados {len(df_ts):,} logs de telemetria.")
else:
    print("✅ Arquivo 'telemetria_servidor.csv' já existe. Pronto para as consultas!")

⏳ Gerando 'telemetria_servidor.csv' no disco (isso pode levar uns 10 a 15 segundos)...
✅ Arquivo salvo! Foram gerados 1,000,000 logs de telemetria.


In [2]:
import time
import pandas as pd
import ipywidgets as widgets
from ipywidgets import interact

# 1. Carregando os dados e preparando o Índice Temporal (Simulando o motor InfluxDB)
print("⏳ Carregando motor Time-Series para a memória...")
df_ts = pd.read_csv('telemetria_servidor.csv')
# O segredo do Time-Series: O tempo vira o índice de busca da estrutura
df_ts['timestamp'] = pd.to_datetime(df_ts['timestamp'])
df_ts.set_index('timestamp', inplace=True)
print("✅ Motor pronto!")

# 2. Função de Visualização
def exibir_resultado(query: str, titulo: str, explicacao: str, tempo_ms: float, registros_lidos: int, resultado: pd.DataFrame):
    print(f"\n{'=' * 80}")
    print("⚙️  PAINEL DE TESTES TIME-SERIES")
    print(f"{'=' * 80}")

    print("\n💻 QUERY (FLUX / SQL SIMULADO)")
    print(f"{'-' * 80}")
    print(query.strip())

    print("\n📊 RESUMO")
    print(f"{'-' * 80}")
    print(f"Tempo de execução : {tempo_ms:.2f} ms")
    print(f"Janela processada : {registros_lidos:,} registros".replace(",", "."))
    print(f"Resultado         : {titulo}")

    print("\n📝 O QUE ACONTECEU?")
    print(f"{'-' * 80}")
    print(explicacao)

    print("\n📋 RESULTADO DA CONSULTA (Amostra / Agregação)")
    print(f"{'-' * 80}")
    
    if resultado.empty:
        print("Nenhum registro encontrado na janela.")
    else:
        print(resultado.head(3).to_string())
        print("...")

    print(f"\n{'=' * 80}")


# 3. Lógica de Execução
def testar_timeseries(tipo_query: str):
    inicio = time.time()

    if tipo_query == "Eficiente (Fatiamento por Janela de Tempo)":
        query = """
                SELECT mean(cpu_usage) FROM telemetria 
                WHERE time >= '2026-01-05 12:00:00' AND time <= '2026-01-05 12:59:59'
                """
        # Busca indexada nativa por fatiamento de tempo
        janela = df_ts.loc['2026-01-05 12:00:00':'2026-01-05 12:59:59']
        resultado = janela.mean().to_frame(name='Média da Hora')
        
        titulo = "🚀 Slicing Cronológico Direto"
        explicacao = (
            "Os dados estão agrupados no disco por blocos de tempo.\n"
            "O banco pulou dias inteiros de dados ignorados e foi cirurgicamente "
            "no bloco daquele dia e hora específicos. É extremamente eficiente."
        )
        registros_lidos = len(janela) # Processou apenas as 3600 linhas daquela hora
        
    else:
        query = """
                SELECT * FROM telemetria 
                WHERE cpu_usage > 99.5
                """
        # Scan global procurando por um valor numérico fora do tempo
        resultado = df_ts[df_ts['cpu_usage'] > 99.5]
        
        titulo = "🚨 Scan Global de Métrica"
        explicacao = (
            "O banco foi forçado a varrer a história inteira (todos os 1 milhão de "
            "segundos dos últimos 11 dias) abrindo bloco por bloco só para checar se "
            "naquele segundo a CPU passou de 99.5%. Time-Series odeiam buscas sem filtro de tempo."
        )
        registros_lidos = len(df_ts) # Leu o milhão de registros

    fim = time.time()
    tempo_ms = (fim - inicio) * 1000

    exibir_resultado(query, titulo, explicacao, tempo_ms, registros_lidos, resultado)


# 4. Interface Gráfica
opcoes_cenarios = [
    "Eficiente (Fatiamento por Janela de Tempo)",
    "Ineficiente (Busca Global por Valor da Métrica)"
]

interact(
    testar_timeseries,
    tipo_query=widgets.RadioButtons(
        options=opcoes_cenarios,
        description="Cenário:",
        layout={'width': 'max-content'}
    )
)

⏳ Carregando motor Time-Series para a memória...
✅ Motor pronto!


interactive(children=(RadioButtons(description='Cenário:', layout=Layout(width='max-content'), options=('Efici…

<function __main__.testar_timeseries(tipo_query: str)>